# BigQuery Studio Visual Edition · Act 2: Customer 360 Subgraph, `GRAPH_EXPAND` & 3-Method Scorecard

> **Designed for BigQuery Studio (Colab Enterprise in the Google Cloud Console).**
> Combines interactive `%%bigquery --graph` Customer 360 visualisations with declarative `GRAPH_EXPAND` + `AGG()` semantic queries and the 3-method benchmark scorecard.

## 1 · Interactive Customer 360 Subgraph for `Abby Noland` (`PER-1cdfdb9f52ddb25a`)

Run the cell below in BigQuery Studio to render the complete multi-hop Customer 360 subgraph around `PER-1cdfdb9f52ddb25a` (`Abby Noland`):
- **5 `SourceRecord` nodes** (`CRM-5d268dd4`, `LOY-ac677db5`, `SUP-da73f0e1`, `POS-1b222aed`, `ENR-31fd9a08`)
- Their shared **`Identifier`**, **`Email`**, **`Phone`**, **`Account`**, **`Address`**, and **`Household`** nodes.

In [1]:
%%bigquery --graph
GRAPH `all-things-cdp.cdp.cdp_identity_graph`
MATCH (i:Identifier)<-[hi:HAS_IDENTIFIER]-(r:SourceRecord)-[res:RESOLVES_TO]->(p:Person)-[la:LIVES_AT]->(a:Address)
WHERE p.person_id = 'PER-1cdfdb9f52ddb25a'
RETURN TO_JSON(i) AS i, TO_JSON(hi) AS hi, TO_JSON(r) AS r, TO_JSON(res) AS res, TO_JSON(p) AS p, TO_JSON(la) AS la, TO_JSON(a) AS a

Rendered Customer 360 subgraph for PER-1cdfdb9f52ddb25a (Abby Noland: 5 SourceRecords across CRM, LOYALTY, SUPPORT, POS, ENRICH).

## 2 · Declarative Semantic Graph Aggregation (`FROM GRAPH_EXPAND`)

Query `all-things-cdp.cdp.cdp_semantic_graph` using `GRAPH_EXPAND` and `AGG()` to compare cluster counts and hairball membership across all 7 retail brands/sources without join fan-out.

In [2]:
%%bigquery
SELECT
  Profile_sources AS brand,
  AGG(Profile_profile_count)                    AS profiles,
  AGG(Membership_link_count)                    AS links,
  AGG(Identifier_identifier_count)              AS distinct_identifiers,
  AGG(Profile_profiles_touching_hubs)           AS touching_hubs,
  AGG(Profile_baseline_wesid_count)             AS baseline_clusters,
  AGG(Profile_weighted_wesid_count)             AS weighted_clusters,
  AGG(Profile_composable_wesid_count)           AS composable_clusters,
  AGG(Profile_profiles_in_baseline_hairballs)   AS in_baseline_hairballs,
  AGG(Profile_profiles_in_weighted_hairballs)   AS in_weighted_hairballs,
  AGG(Profile_profiles_in_composable_hairballs) AS in_composable_hairballs
FROM GRAPH_EXPAND("all-things-cdp.cdp.cdp_semantic_graph")
GROUP BY brand
ORDER BY profiles DESC

brand,profiles,links,distinct_identifiers,touching_hubs,baseline_clusters,weighted_clusters,composable_clusters,in_baseline_hairballs,in_weighted_hairballs,in_composable_hairballs
CRM,4155,9085,8295,679,2948,4133,4154,1196,0,0
ECOM,3752,6110,5386,628,2733,3730,3751,1008,0,0
LOYALTY,3248,7095,6503,525,2287,3248,3246,962,0,0
POS,2516,2956,1665,927,1189,2255,2252,1180,0,0
ENRICH,1842,2165,1711,291,1268,1704,1718,486,0,0
SUPPORT,854,1148,979,132,568,823,819,265,0,0
CALL,257,347,277,66,160,257,257,98,0,0


## 3 · Side-by-Side 3-Method Benchmark (`all-things-cdp.cdp.v_method_comparison`)

In [3]:
%%bigquery
SELECT
  pathology,
  method,
  tp,
  fp,
  fn,
  ROUND(precision, 4) AS precision,
  ROUND(recall, 4)    AS recall,
  ROUND(f1, 4)        AS f1
FROM `all-things-cdp.cdp.v_method_comparison`
WHERE pathology IN ('ALL', 'hub', 'household', 'supersession', 'semantic_fuzzy')
ORDER BY
  CASE pathology
    WHEN 'ALL' THEN 0 WHEN 'hub' THEN 1 WHEN 'household' THEN 2
    WHEN 'supersession' THEN 3 WHEN 'semantic_fuzzy' THEN 4
  END,
  CASE method WHEN 'baseline_cc' THEN 1 WHEN 'weighted_cc' THEN 2 ELSE 3 END

pathology,method,tp,fp,fn,precision,recall,f1
ALL,baseline_cc,8064,13488214,10712,0.0006,0.4295,0.0012
ALL,weighted_cc,7566,321,11210,0.9593,0.403,0.5675
ALL,composable_cdp,8782,74,9994,0.9916,0.4677,0.6356
hub,baseline_cc,2557,13488039,2883,0.0002,0.47,0.0004
hub,weighted_cc,2077,64,3363,0.9701,0.3818,0.5479
hub,composable_cdp,2404,19,3036,0.9922,0.4419,0.6115
household,baseline_cc,151,54,87,0.7366,0.6345,0.6817
household,weighted_cc,147,100,91,0.5951,0.6176,0.6062
household,composable_cdp,152,2,86,0.987,0.6387,0.7755
supersession,baseline_cc,28,29,62,0.4912,0.3111,0.381
